In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")


In [8]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS


In [ ]:
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain import hub


In [9]:
pdf_path = "./msa.pdf"


# 1. Data Ingestion

In [11]:
loader = PyPDFLoader(file_path=pdf_path)
documents = loader.load()


In [ ]:
from langchain_community.document_loaders import TextLoader
# Load text data from a file
data_loader = TextLoader("data.txt")
documents = data_loader.load()
print("--------documents---------", documents)


# 2. Chunking

In [12]:

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=30, separator="\n")
split_documents = text_splitter.split_documents(documents)


In [ ]:
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# # Define a chunking strategy
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=500, chunk_overlap=50
# )
# chunks = text_splitter.split_documents(documents)


In [13]:
chunks = split_documents

In [14]:
chunks

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2023-09-13T16:24:52+05:30', 'author': '101498', 'moddate': '2023-09-13T16:24:52+05:30', 'title': 'Microsoft Word - Draft agreement', 'source': './msa.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1'}, page_content='Master Service Agreement \nThis Agreement is made at this  ……… day of ___________ ,202__ at New Delhi. \n \n                                                 BY AND BETWEEN \n \nINDIA POST PAYMENTS BANK LIMITED,  a public limited company wholly owned by the \nGovernment of India through Department of Post under Ministry of Communication and set up \nunder the Companies Act, 2013, and the Banking Regulation Act, 1949 as a Payments Bank \nunder the Department of Posts and in line with relevant guidelines of the Reserve Bank of India, \nhaving its Registered & Corporate Office at Post Office, Speed Post Centre Building, Market \nRoad, New Delhi – 110001 (hereinafter referred to a

# 3. Store Embeddings in a FAISS Vector Store

In [24]:
from langchain_classic.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import pickle

# Load Sentence Transformer embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# Create FAISS vector store
vector_store = FAISS.from_documents(chunks, embeddings)

# Save FAISS index and document mapping
vector_store.save_local("faiss_index")

# Store documents separately
with open("faiss_docs.pkl", "wb") as f:
    pickle.dump(documents, f)

print("FAISS index and documents stored successfully.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21597.75it/s]


FAISS index and documents stored successfully.


In [ ]:
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(split_documents, embeddings)

# Save the vector store
vectorstore.save("faiss_index")

# Load the vector store
new_vectorstore = FAISS.load_local(
       "faiss_index_react", embeddings, allow_dangerous_deserialization=True
   )


In [25]:
# Load the vector store
new_vectorstore = FAISS.load_local(
       "faiss_index_react", embeddings, allow_dangerous_deserialization=True
   )

RuntimeError: Error in faiss::FileIOReader::FileIOReader(const char *) at /Users/runner/work/faiss/faiss/faiss/impl/io.cpp:70: Error: 'f' failed: could not open faiss_index_react/index.faiss for reading: No such file or directory

# 4. Query Retrieval & Answer Generation with Google Gemini


In [ ]:
import google.generativeai as genai
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# Load FAISS index
vector_store = FAISS.load_local("faiss_index", embedding_model, allow_dangerous_deserialization=True)

# Initialize retriever
retriever = vector_store.as_retriever()

# Configure Gemini API
genai.configure(api_key=GEMINI_API_KEY)
def query_gemini(query, context):
    """Uses Gemini Pro to answer questions based on retrieved context."""
    model = genai.GenerativeModel("gemini-2.5-flash-lite")
    prompt = f"Answer the following question based on the provided context:\n\nContext: {context}\n\nQuestion: {query}"
    response = model.generate_content(prompt)
    return response.text

# Perform Retrieval & Answer Generation
query = "What is NLP?"
retrieved_docs = retriever.get_relevant_documents(query)
context = "\n".join([doc.page_content for doc in retrieved_docs])

# Generate response using Gemini
response = query_gemini(query, context)
print(f"\n🔹 **Q:** {query}\n🔹 **A:** {response}")


In [ ]:
retrieval_qa_chat_prompt = hub.pull("langchain-ai/retrieval-qa-chat")

combine_docs_chain = create_stuff_documents_chain(
       OpenAI(), retrieval_qa_chat_prompt
   )

retrieval_chain = create_retrieval_chain(
       new_vectorstore.as_retriever(), combine_docs_chain
   )


In [ ]:
res = retrieval_chain.invoke({"input": "Give me the gist of Retrieval-Augmented Generation (RAG) in 3 sentences"})
print(res["answer"])


# LLM as a judge